# ResNet50 FT-V2 Error Analysis, Calibration, And Inference

This notebook focuses on the current project champion, **ResNet50 FT-V2**. It does not start a new architecture search. Instead, it evaluates confidence quality, hard-class behavior, high-confidence mistakes, and deterministic single-image inference.


## 1. Experiment Question

The previous notebooks showed that **ResNet50 FT-V2** is the best current model. The next question is:

> Can we make the champion easier to trust and safer to use in a real product workflow?

This notebook answers that by measuring **calibration**, inspecting **error clusters**, exporting **actionable diagnostics**, and providing a clean inference helper.


In [ ]:
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm


In [ ]:
@dataclass(frozen=True)
class CFG:
    """Configuration for champion analysis and inference."""

    SEED: int = 42
    BATCH_SIZE: int = 32
    IMAGE_SIZE: tuple[int, int] = (224, 224)
    NUM_CLASSES: int = 101
    NUM_WORKERS: int = 0
    TOP_K: int = 5
    ECE_BINS: int = 15
    HARD_CLASS_COUNT: int = 15
    ERROR_EXAMPLE_COUNT: int = 20
    DATA_DIR: Path = Path("/kaggle/input/datasets/kmader/food41")
    ARTIFACT_DIR: Path = Path(
        "/kaggle/input/models/tuannm3823/food101-resnet50-refinements/"
        "pytorch/default/1"
    )
    CHECKPOINT_NAME: str = "resnet50_ft_v2_best.pth"
    RESULTS_DIR: Path = Path("/kaggle/working/results/resnet50_error_calibration")
    SAMPLE_IMAGE_PATH: str | None = None


CFG.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
random.seed(CFG.SEED)
np.random.seed(CFG.SEED)
torch.manual_seed(CFG.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Artifact directory: {CFG.ARTIFACT_DIR}")


## 2. Data And Split

The notebook keeps the same stratified split contract as the earlier experiments. This makes calibration and error reports comparable with the champion metrics already recorded in the project.


In [ ]:
def resolve_image_dir(data_dir: Path) -> Path:
    """Resolve the Food-101 image directory from the configured dataset root."""
    candidate_dirs = [data_dir / "images", data_dir]
    for candidate_dir in candidate_dirs:
        if not candidate_dir.exists():
            continue
        class_dirs = [path for path in candidate_dir.iterdir() if path.is_dir()]
        has_images = any(
            image_path.suffix.lower() in {".jpg", ".jpeg", ".png"}
            for class_dir in class_dirs
            for image_path in class_dir.iterdir()
        )
        if has_images:
            return candidate_dir
    raise FileNotFoundError(
        "Food-101 images were not found under "
        f"{data_dir / 'images'} or directly under {data_dir}."
    )


def create_data_manifest(image_dir: Path) -> pd.DataFrame:
    """Create an image-path and label manifest from class folders."""
    records: list[dict[str, str]] = []
    for class_dir in sorted(path for path in image_dir.iterdir() if path.is_dir()):
        for image_path in sorted(class_dir.iterdir()):
            if image_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                records.append({"path": str(image_path), "label": class_dir.name})
    if not records:
        raise ValueError(f"No image files were found under {image_dir}.")
    return pd.DataFrame.from_records(records, columns=["path", "label"])


def split_manifest(manifest: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Create reproducible stratified train, validation, and test splits."""
    train_df, temp_df = train_test_split(
        manifest,
        test_size=0.2,
        random_state=CFG.SEED,
        stratify=manifest["label"],
    )
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,
        random_state=CFG.SEED,
        stratify=temp_df["label"],
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


IMAGE_DIR = resolve_image_dir(CFG.DATA_DIR)
df = create_data_manifest(IMAGE_DIR)
train_df, val_df, test_df = split_manifest(df)
class_names = sorted(df["label"].unique())
class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}

print(f"Image directory: {IMAGE_DIR}")
print(f"Images: {len(df):,}")
print(f"Classes: {len(class_names):,}")
print(f"Train/val/test: {len(train_df):,} / {len(val_df):,} / {len(test_df):,}")


In [ ]:
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

EVAL_TRANSFORMS = transforms.Compose(
    [
        transforms.Resize(CFG.IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(NORM_MEAN, NORM_STD),
    ]
)


class FoodDataset(Dataset):
    """PyTorch dataset for Food-101 image classification."""

    def __init__(self, dataframe: pd.DataFrame, class_to_idx: dict[str, int], transform=None) -> None:
        self.df = dataframe.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:
        row = self.df.iloc[index]
        image = Image.open(row["path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, self.class_to_idx[row["label"]]


def create_loader(dataframe: pd.DataFrame) -> DataLoader:
    """Create an evaluation DataLoader."""
    return DataLoader(
        FoodDataset(dataframe, class_to_idx, EVAL_TRANSFORMS),
        batch_size=CFG.BATCH_SIZE,
        shuffle=False,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=device.type == "cuda",
    )


val_loader = create_loader(val_df)
test_loader = create_loader(test_df)


## 3. Champion Model

The model architecture matches the ResNet50 FT-V2 checkpoint produced by Notebook 2. The checkpoint is loaded from the Kaggle Model artifact path.


In [ ]:
def make_classifier_head(in_features: int) -> nn.Sequential:
    """Create the project-standard Food-101 classifier head."""
    return nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, CFG.NUM_CLASSES),
    )


def build_resnet50() -> nn.Module:
    """Build ResNet50 with the project classifier head."""
    model = models.resnet50(weights=None)
    model.fc = make_classifier_head(model.fc.in_features)
    return model


def resolve_checkpoint() -> Path:
    """Resolve the champion checkpoint path."""
    candidates = [
        CFG.ARTIFACT_DIR / CFG.CHECKPOINT_NAME,
        CFG.ARTIFACT_DIR / "food101-resnet50-refinements" / CFG.CHECKPOINT_NAME,
        CFG.RESULTS_DIR / CFG.CHECKPOINT_NAME,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    if CFG.ARTIFACT_DIR.exists():
        matches = sorted(CFG.ARTIFACT_DIR.rglob(CFG.CHECKPOINT_NAME))
        if matches:
            return matches[0]
    searched = "\n".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f"Could not find {CFG.CHECKPOINT_NAME}. Searched:\n{searched}")


checkpoint_path = resolve_checkpoint()
model = build_resnet50()
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model = model.to(device)
model.eval()
print(f"Loaded checkpoint: {checkpoint_path}")


## 4. Prediction Collection

Collect logits once, then reuse them for accuracy, calibration, confusion, and error reports. This avoids repeated model passes and keeps the diagnostics consistent.


In [ ]:
def collect_logits(model: nn.Module, loader: DataLoader) -> tuple[torch.Tensor, torch.Tensor, pd.DataFrame]:
    """Collect logits, labels, and the aligned manifest for a loader."""
    model.eval()
    logits_batches = []
    label_batches = []

    with torch.no_grad():
        for images, labels in tqdm(loader, leave=False):
            images = images.to(device)
            logits = model(images).cpu()
            logits_batches.append(logits)
            label_batches.append(labels.cpu())

    return torch.cat(logits_batches), torch.cat(label_batches), loader.dataset.df.reset_index(drop=True)


def prediction_frame(logits: torch.Tensor, labels: torch.Tensor, manifest: pd.DataFrame, temperature: float = 1.0) -> pd.DataFrame:
    """Create a prediction table from logits and labels."""
    probabilities = F.softmax(logits / temperature, dim=1)
    top_probabilities, top_indices = probabilities.topk(CFG.TOP_K, dim=1)
    rows = []
    for idx, label_idx in enumerate(labels.tolist()):
        predicted_idx = top_indices[idx, 0].item()
        rows.append(
            {
                "path": manifest.loc[idx, "path"],
                "actual": class_names[label_idx],
                "predicted": class_names[predicted_idx],
                "confidence": top_probabilities[idx, 0].item(),
                "is_correct": predicted_idx == label_idx,
                "top_5": "|".join(class_names[class_idx] for class_idx in top_indices[idx].tolist()),
                "top_5_confidence": "|".join(f"{prob:.6f}" for prob in top_probabilities[idx].tolist()),
            }
        )
    return pd.DataFrame(rows)


def metric_frame(predictions_df: pd.DataFrame) -> pd.DataFrame:
    """Summarize top-1 and top-5 accuracy."""
    top_1 = predictions_df["is_correct"].mean()
    top_5 = predictions_df.apply(lambda row: row["actual"] in row["top_5"].split("|"), axis=1).mean()
    return pd.DataFrame(
        [
            {"metric": "top_1_accuracy", "value": top_1},
            {"metric": "top_5_accuracy", "value": top_5},
        ]
    )


val_logits, val_labels, val_manifest = collect_logits(model, val_loader)
test_logits, test_labels, test_manifest = collect_logits(model, test_loader)


## 5. Calibration Analysis

Temperature scaling learns one scalar on validation logits. It does not change top-k ranking, but it can make confidence values better aligned with actual correctness.


In [ ]:
def expected_calibration_error(probabilities: torch.Tensor, labels: torch.Tensor, n_bins: int = CFG.ECE_BINS) -> float:
    """Compute expected calibration error for top-1 confidence."""
    confidences, predictions = probabilities.max(dim=1)
    accuracies = predictions.eq(labels)
    bin_boundaries = torch.linspace(0, 1, n_bins + 1)
    ece = torch.zeros(1)

    for lower_bound, upper_bound in zip(bin_boundaries[:-1], bin_boundaries[1:]):
        in_bin = confidences.gt(lower_bound) & confidences.le(upper_bound)
        proportion = in_bin.float().mean()
        if proportion.item() > 0:
            accuracy_in_bin = accuracies[in_bin].float().mean()
            confidence_in_bin = confidences[in_bin].mean()
            ece += torch.abs(confidence_in_bin - accuracy_in_bin) * proportion
    return ece.item()


def calibration_bins(probabilities: torch.Tensor, labels: torch.Tensor, n_bins: int = CFG.ECE_BINS) -> pd.DataFrame:
    """Create a calibration-bin summary."""
    confidences, predictions = probabilities.max(dim=1)
    correct = predictions.eq(labels)
    rows = []
    bin_boundaries = torch.linspace(0, 1, n_bins + 1)
    for lower_bound, upper_bound in zip(bin_boundaries[:-1], bin_boundaries[1:]):
        in_bin = confidences.gt(lower_bound) & confidences.le(upper_bound)
        if in_bin.sum().item() == 0:
            continue
        rows.append(
            {
                "bin_lower": lower_bound.item(),
                "bin_upper": upper_bound.item(),
                "sample_count": int(in_bin.sum().item()),
                "accuracy": correct[in_bin].float().mean().item(),
                "mean_confidence": confidences[in_bin].mean().item(),
            }
        )
    return pd.DataFrame(rows)


def learn_temperature(logits: torch.Tensor, labels: torch.Tensor) -> float:
    """Fit a scalar temperature on validation logits."""
    temperature = nn.Parameter(torch.ones(1))
    optimizer = optim.LBFGS([temperature], lr=0.01, max_iter=50)
    criterion = nn.CrossEntropyLoss()

    def closure():
        optimizer.zero_grad()
        loss = criterion(logits / temperature.clamp(min=0.05), labels)
        loss.backward()
        return loss

    optimizer.step(closure)
    return temperature.detach().clamp(min=0.05).item()


temperature = learn_temperature(val_logits, val_labels)
val_probabilities_raw = F.softmax(val_logits, dim=1)
val_probabilities_calibrated = F.softmax(val_logits / temperature, dim=1)
test_probabilities_raw = F.softmax(test_logits, dim=1)
test_probabilities_calibrated = F.softmax(test_logits / temperature, dim=1)

calibration_summary = pd.DataFrame(
    [
        {
            "split": "validation",
            "temperature": temperature,
            "ece_before": expected_calibration_error(val_probabilities_raw, val_labels),
            "ece_after": expected_calibration_error(val_probabilities_calibrated, val_labels),
        },
        {
            "split": "test",
            "temperature": temperature,
            "ece_before": expected_calibration_error(test_probabilities_raw, test_labels),
            "ece_after": expected_calibration_error(test_probabilities_calibrated, test_labels),
        },
    ]
)
calibration_summary.to_csv(CFG.RESULTS_DIR / "calibration_summary.csv", index=False)
calibration_bins(test_probabilities_raw, test_labels).to_csv(CFG.RESULTS_DIR / "test_calibration_bins_raw.csv", index=False)
calibration_bins(test_probabilities_calibrated, test_labels).to_csv(CFG.RESULTS_DIR / "test_calibration_bins_calibrated.csv", index=False)
calibration_summary


## 6. Error Reports

Export hard classes, confusion pairs, and high-confidence wrong predictions. These outputs should guide the next product and modeling decisions.


In [ ]:
val_predictions = prediction_frame(val_logits, val_labels, val_manifest, temperature=temperature)
test_predictions = prediction_frame(test_logits, test_labels, test_manifest, temperature=temperature)
val_metrics = metric_frame(val_predictions)
test_metrics = metric_frame(test_predictions)

val_predictions.to_csv(CFG.RESULTS_DIR / "val_predictions_calibrated.csv", index=False)
test_predictions.to_csv(CFG.RESULTS_DIR / "test_predictions_calibrated.csv", index=False)
val_metrics.to_csv(CFG.RESULTS_DIR / "val_metrics_calibrated.csv", index=False)
test_metrics.to_csv(CFG.RESULTS_DIR / "test_metrics_calibrated.csv", index=False)

test_report = pd.DataFrame(
    classification_report(
        test_predictions["actual"],
        test_predictions["predicted"],
        output_dict=True,
        zero_division=0,
    )
).T.reset_index(names="class_name")
test_report.to_csv(CFG.RESULTS_DIR / "test_class_report_calibrated.csv", index=False)

hard_classes = (
    test_report[test_report["class_name"].isin(class_names)]
    .sort_values("f1-score")
    .head(CFG.HARD_CLASS_COUNT)
)
hard_classes.to_csv(CFG.RESULTS_DIR / "hard_classes_calibrated.csv", index=False)

confusion_pairs = (
    test_predictions[~test_predictions["is_correct"]]
    .groupby(["actual", "predicted"], as_index=False)
    .agg(count=("path", "count"), mean_confidence=("confidence", "mean"))
    .sort_values(["count", "mean_confidence"], ascending=False)
    .head(30)
)
confusion_pairs.to_csv(CFG.RESULTS_DIR / "top_confusion_pairs_calibrated.csv", index=False)

high_confidence_errors = (
    test_predictions[~test_predictions["is_correct"]]
    .sort_values("confidence", ascending=False)
    .head(CFG.ERROR_EXAMPLE_COUNT)
)
high_confidence_errors.to_csv(CFG.RESULTS_DIR / "high_confidence_errors_calibrated.csv", index=False)

print("Test metrics")
display(test_metrics)
print("Hardest classes")
display(hard_classes[["class_name", "precision", "recall", "f1-score", "support"]])
print("Top confusion pairs")
display(confusion_pairs)
print("High-confidence errors")
display(high_confidence_errors[["actual", "predicted", "confidence", "path"]])


## 7. Deterministic Single-Image Inference

Use this helper for deployment-style checks. It applies the same preprocessing and optional calibrated temperature used by the evaluation cells.


In [ ]:
def predict_food(image_path: str | Path, model: nn.Module, temperature: float = temperature, top_k: int = CFG.TOP_K) -> pd.DataFrame:
    """Predict top-k Food-101 classes for one image."""
    image_path = Path(image_path)
    image = Image.open(image_path).convert("RGB")
    image_tensor = EVAL_TRANSFORMS(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(image_tensor).cpu()
        probabilities = F.softmax(logits / temperature, dim=1)
        top_probabilities, top_indices = probabilities.topk(top_k, dim=1)

    return pd.DataFrame(
        [
            {
                "rank": rank + 1,
                "class_name": class_names[class_idx],
                "confidence": confidence,
            }
            for rank, (class_idx, confidence) in enumerate(
                zip(top_indices[0].tolist(), top_probabilities[0].tolist())
            )
        ]
    )


if CFG.SAMPLE_IMAGE_PATH:
    display(predict_food(CFG.SAMPLE_IMAGE_PATH, model))
else:
    example_path = test_manifest.loc[0, "path"]
    print(f"Example image: {example_path}")
    display(predict_food(example_path, model))


## 8. Run Insight

Use the exported calibration and error-analysis files to decide whether the model is ready for a user-facing prototype. If calibration improves ECE without changing accuracy, the calibrated confidence values should be preferred for product messaging and review thresholds.
